# Etapa 2 — Estoque Projetado e Análise de Cobertura *(versão corrigida)*

**Case Técnico — Análise de Desempenho de Produtos no Varejo**

## Objetivos
1. Projetar o estoque mês a mês: `Estoque_t = Estoque_inicial + ΣEntradas − ΣSaídas`
2. Identificar pares loja×produto em **ruptura** (estoque ≤ 0) em dez/2025
3. Calcular **dias de cobertura** por par
4. Classificar o portfólio: *Em Ruptura / Crítico / Atenção / Saudável / Sem Venda*
5. Priorizar por **receita histórica** os itens que exigem reposição
6. Investigar **outliers de preço** e documentar a causa

## Premissas metodológicas
- Tudo em **unidade de armazenagem**: vendas × `CONVERSAO_VENDA_PARA_ARMAZENAGEM`,
  compras × `CONVERSAO_COMPRA_ARMAZENAGEM`.
- **Loja 93 (atacado)**: a separação atacado×rede é um **filtro posterior** de análise,
  nunca uma remoção de dados antes de calcular receita ou estoque.
- **Estoque negativo = ruptura** (saídas acima do estoque visível): direção conservadora.

## Revisão de qualidade — 4 correções aplicadas nesta versão
| # | Severidade | Problema | Correção |
|---|---|---|---|
| **1** | Crítico | Receita por par calculada **sem a loja 93** → 263 pares da loja 93 em ruptura ficavam com receita nula e sumiam do ranking | Calcular receita com **todas as lojas** (`excluir_atacado=False`) |
| **2** | Crítico | Skeleton montado **só com o estoque inicial** → 3.379 pares com venda e sem estoque (R$ 87,5M) eram ignorados | Skeleton = **união** de estoque ∪ vendas ∪ compras; sem foto inicial → `ESTOQUE_INICIAL = 0` |
| **3** | Médio | `DIAS_COBERTURA` ficava **negativo** (ex.: −566) em rupturas | Piso em 0 quando `ESTOQUE_PROJ ≤ 0` |
| **4** | Médio | Preço de um produto variando de R$3,68 a R$391,55 na mesma loja | Investigado: venda em **caixa** (EMBALAGEM=1) com conversão aplicada — **não é bug** |

> Cada correção abaixo vem com uma **célula de validação antes/depois**.

<!-- glossario-comercial -->
## 📖 Glossário rápido (para leitura não técnica)

Esta seção traduz os termos técnicos do notebook para linguagem de negócio. Quem é de áreas como
comercial, marketing ou compras pode ler os resultados sem precisar do detalhe técnico.

| Termo | O que significa, em linguagem comercial |
|---|---|
| **SKU** | Código único de um produto. Cada item distinto do catálogo é um SKU (cor/tamanho/voltagem diferentes contam como SKUs diferentes). |
| **Par loja × produto** | Um produto específico dentro de uma loja específica — o menor nível de detalhe da análise de estoque. O mesmo produto em duas lojas são dois pares. |
| **Unidade de armazenagem** | A unidade padrão de contagem do estoque (p.ex. a caixa). Convertemos tudo para ela para não somar "3 caixas" com "5 unidades soltas" como se fossem iguais. |
| **Estoque projetado** | Estimativa do saldo de cada item, reconstruída pela conta *estoque de abertura + compras − vendas*. **Não é contagem física** de prateleira. |
| **Snapshot (foto)** | A situação do estoque num momento específico — aqui, o fim do período (dez/2025). |
| **Cobertura (dias de estoque)** | Por quantos dias o estoque atual daria conta da venda média. Pouca cobertura = risco de faltar produto em breve. |
| **Ruptura** | Item com estoque estimado zerado ou negativo — sem saldo para vender segundo o histórico disponível. |
| **Receita em risco** | Quanto de receita esses itens **já geraram no passado**. Serve para ordenar a fila de reposição (repor primeiro o que mais vende) — **não é perda prevista**. |
| **Curva ABC** | Regra de Pareto aplicada à receita: a **classe A** são os poucos produtos que somam ~80% do faturamento; B e C têm peso decrescente. |
| **Rede física × atacado/B2B (Loja 93)** | A Loja 93 vende em grande volume para outras empresas (atacado), com ticket ~20× o das demais. Por isso é mostrada à parte, para não distorcer a média das lojas de varejo. |
| **Linhas de venda (TRANSACOES)** | Cada linha de item vendido. Não é o número de cupons/notas (a base não traz id de cupom) — por isso é uma medida aproximada. |
| **Proxy** | Uma medida aproximada usada quando o dado ideal não existe na base (ex.: usar linhas de venda no lugar de número de cupons). |
| **Variação ano contra ano (YoY)** | Comparação de um período com o mesmo período do ano anterior (ex.: 2025 vs 2024), para isolar sazonalidade. |
| **Conservador (por construção)** | Na dúvida, o método assume o pior cenário (falta de estoque). Erra para **alertar a mais**, nunca a menos — escolha de segurança. |
| **Parquet** | Formato de arquivo compactado e rápido para grandes volumes de dados, usado internamente para acelerar o processamento. |
| **Skeleton (malha base)** | A grade inicial com todas as combinações loja × produto × mês, montada **antes** de preencher vendas e compras — garante que nenhum item suma da conta de estoque. |


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

_p = Path.cwd()
while not (_p / "src" / "utils.py").exists() and _p != _p.parent:
    _p = _p.parent
ROOT = _p
sys.path.insert(0, str(ROOT / "src"))

from utils import (PROCESSED, OUTPUTS, LOJA_ATACADO,
                   load_vendas, load_compras, load_estoque_inicial,
                   load_dim_produto, load_dim_lojas)
OUT = OUTPUTS / "etapa2"; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 170)

# Bug 1: carregamos as vendas com TODAS as lojas (a loja 93 também movimenta estoque)
v = load_vendas(excluir_atacado=False)
c = load_compras()
e = load_estoque_inicial()
p = load_dim_produto()
l = load_dim_lojas()
print(f"vendas: {len(v):,} | compras: {len(c):,} | estoque inicial: {len(e):,} pares-loja")

vendas: 1,090,390 | compras: 1,393 | estoque inicial: 25,330 pares-loja


**Por quê carregar a loja 93 aqui?** A loja 93 está no `fato_estoque_inicial` e gera receita
real. Removê-la *antes* dos cálculos (como na versão com bug) zerava indicadores de pares
válidos. Carregamos tudo e segregamos o atacado só onde a análise pede (cobertura da rede física).

In [2]:
# 2 — Compras → unidade de armazenagem
conv_map = p.set_index("CODIGO")["CONVERSAO_COMPRA_ARMAZENAGEM"]
c["CONVERSAO_COMPRA_ARM"] = c["CODIGO"].map(conv_map).fillna(1.0)
c["QTD_COMPRA_ARM"]       = c["QUANTIDADE_COMPRA"] * c["CONVERSAO_COMPRA_ARM"]
c[["CODIGO","QUANTIDADE_COMPRA","CONVERSAO_COMPRA_ARM","QTD_COMPRA_ARM"]].head(4)

,CODIGO,QUANTIDADE_COMPRA,CONVERSAO_COMPRA_ARM,QTD_COMPRA_ARM
0,152965,12.0,1.0,12.0
1,152965,6.0,1.0,6.0
2,29896,120.0,1.0,120.0
3,29896,80.0,1.0,80.0


**Por quê?** Compras vêm na unidade do fornecedor (caixas). Para somar com estoque e saídas
(ambos em unidade de armazenagem), convertemos `QUANTIDADE_COMPRA × fator`.

In [3]:
# 3 — Movimentos mensais (saídas = vendas, entradas = compras)
v["ANO_MES_DT"] = v["DATA_VENDA"].dt.to_period("M")
c["ANO_MES_DT"] = c["DATA_ENTRADA"].dt.to_period("M")

saidas_mensais = (v.groupby(["COD_EMPRESA","CODIGO","ANO_MES_DT"])["QTD_ARMAZENAGEM"].sum()
                  .reset_index().rename(columns={"QTD_ARMAZENAGEM":"SAIDA_MES"}))
entradas_mensais = (c.groupby(["COD_EMPRESA","CODIGO","ANO_MES_DT"])["QTD_COMPRA_ARM"].sum()
                    .reset_index().rename(columns={"QTD_COMPRA_ARM":"ENTRADA_MES"}))
print("saídas mensais :", f"{len(saidas_mensais):,} linhas")
print("entradas mensais:", f"{len(entradas_mensais):,} linhas")

saídas mensais : 213,007 linhas
entradas mensais: 887 linhas


**Por quê agregar por mês?** A série mensal `Estoque_t` precisa de fluxos mensais. Agregamos
saídas e entradas por (loja, produto, mês) antes de montar a grade temporal.

### Correção do Bug 2 — skeleton = união de estoque ∪ vendas ∪ compras
A versão com bug montava o universo de pares **só** com o estoque inicial, deixando de fora
3.379 pares que têm venda mas nenhuma foto de estoque. A célula abaixo demonstra o ganho.

In [4]:
# 4 — Skeleton (Bug 2): UNIÃO de pares de estoque + vendas + compras
todos_meses = pd.period_range("2024-01", "2025-12", freq="M")

pares_so_estoque = e[["COD_EMPRESA","CODIGO"]].drop_duplicates()
pares = (pd.concat([e[["COD_EMPRESA","CODIGO"]],
                    v[["COD_EMPRESA","CODIGO"]],
                    c[["COD_EMPRESA","CODIGO"]]], ignore_index=True)
         .drop_duplicates().reset_index(drop=True))

# --- validação Bug 2: quantos pares e quanta receita seriam perdidos com o skeleton antigo ---
se = set(map(tuple, pares_so_estoque.values))
sv = set(map(tuple, v[["COD_EMPRESA","CODIGO"]].drop_duplicates().values))
pares_venda_sem_estoque = sv - se
v["_par"] = list(zip(v["COD_EMPRESA"], v["CODIGO"]))
rec_perdida = v[v["_par"].isin(pares_venda_sem_estoque)]["RECEITA"].sum()

print(f"Skeleton ANTIGO (só estoque inicial): {len(pares_so_estoque):,} pares")
print(f"Skeleton CORRIGIDO (união)          : {len(pares):,} pares  (+{len(pares)-len(pares_so_estoque):,})")
print(f"Pares com venda e SEM estoque inicial: {len(pares_venda_sem_estoque):,}")
print(f"Receita que ficava invisível        : R$ {rec_perdida/1e6:.1f}M "
      f"({rec_perdida/v['RECEITA'].sum()*100:.1f}% da receita total)")

skeleton = (pares.assign(key=1)
            .merge(pd.DataFrame({"ANO_MES_DT": todos_meses, "key":1}), on="key")
            .drop("key", axis=1))

Skeleton ANTIGO (só estoque inicial): 25,330 pares
Skeleton CORRIGIDO (união)          : 28,721 pares  (+3,391)
Pares com venda e SEM estoque inicial: 3,379
Receita que ficava invisível        : R$ 87.5M (18.1% da receita total)


**Por quê unir as três fontes?** Um par com venda e sem foto de estoque inicial é justamente o
caso mais perigoso (vende e pode estar em ruptura). Pares sem foto inicial entram com
`ESTOQUE_INICIAL = 0` — a mesma lógica conservadora da Etapa 1. **Resultado: +3.379 pares e
+R$ 87,5M (18,1% da receita) voltam ao radar.**

In [5]:
# 5 — Estoque projetado (série mensal)
df = (skeleton
    .merge(saidas_mensais,   on=["COD_EMPRESA","CODIGO","ANO_MES_DT"], how="left")
    .merge(entradas_mensais, on=["COD_EMPRESA","CODIGO","ANO_MES_DT"], how="left")
    .fillna({"SAIDA_MES":0.0, "ENTRADA_MES":0.0})
    .merge(e, on=["COD_EMPRESA","CODIGO"], how="left")
    .fillna({"ESTOQUE_INICIAL":0.0})           # pares sem foto inicial → 0 (Bug 2)
    .sort_values(["COD_EMPRESA","CODIGO","ANO_MES_DT"]).reset_index(drop=True))

df["MOVIMENTACAO"] = df["ENTRADA_MES"] - df["SAIDA_MES"]
df["MOVIM_ACUM"]   = df.groupby(["COD_EMPRESA","CODIGO"])["MOVIMENTACAO"].cumsum()
df["ESTOQUE_PROJ"] = df["ESTOQUE_INICIAL"] + df["MOVIM_ACUM"]
df["ANO_MES_STR"]  = df["ANO_MES_DT"].astype(str)
df = (df.merge(p[["CODIGO","DESCRICAO","NIVEL_1","NIVEL_2","UNIDADE_ESTOQUE"]], on="CODIGO", how="left")
        .merge(l, on="COD_EMPRESA", how="left"))
print(f"Série projetada: {len(df):,} linhas ({df['COD_EMPRESA'].nunique()} lojas × {df['CODIGO'].nunique()} produtos × 24 meses)")
df[df["CODIGO"]==df["CODIGO"].iloc[0]][["COD_EMPRESA","CODIGO","ANO_MES_STR","ENTRADA_MES","SAIDA_MES","ESTOQUE_PROJ"]].head(4)

Série projetada: 689,304 linhas (11 lojas × 2731 produtos × 24 meses)


,COD_EMPRESA,CODIGO,ANO_MES_STR,ENTRADA_MES,SAIDA_MES,ESTOQUE_PROJ
0,1,544,2024-01,0.0,0.0,0.0
1,1,544,2024-02,0.0,0.0,0.0
2,1,544,2024-03,0.0,0.0,0.0
3,1,544,2024-04,0.0,0.0,0.0


**Por quê acumular?** O estoque de um mês é o inicial mais a soma acumulada de
(entradas − saídas). `cumsum()` por par produz a trajetória mês a mês — base do snapshot de dez/25.

### Correção do Bug 3 — `DIAS_COBERTURA` com piso em 0
Cobertura em dias é `Estoque / Venda_média × 30`. Para pares em ruptura (estoque ≤ 0) isso
gerava valores **negativos** sem sentido de negócio. A célula mostra o antes/depois do piso.

In [6]:
# 6 — Snapshot de cobertura (dez/2025)
# Venda média da REDE FÍSICA (loja 93 excluída de propósito — referência de consumo da rede)
venda_mensal_media = (v[v["COD_EMPRESA"] != LOJA_ATACADO]
    .groupby(["COD_EMPRESA","CODIGO","ANO_MES_DT"])["QTD_ARMAZENAGEM"].sum()
    .groupby(["COD_EMPRESA","CODIGO"]).mean().reset_index()
    .rename(columns={"QTD_ARMAZENAGEM":"VENDA_MEDIA_MES"}))

estoque_final = df[df["ANO_MES_DT"]==pd.Period("2025-12","M")][["COD_EMPRESA","CODIGO","ESTOQUE_PROJ"]].copy()
cobertura = estoque_final.merge(venda_mensal_media, on=["COD_EMPRESA","CODIGO"], how="left")
cobertura["VENDA_MEDIA_MES"] = cobertura["VENDA_MEDIA_MES"].fillna(0)

cobertura["DIAS_COBERTURA"] = np.where(cobertura["VENDA_MEDIA_MES"] > 0,
                                       cobertura["ESTOQUE_PROJ"]/cobertura["VENDA_MEDIA_MES"]*30, np.inf)

# --- validação Bug 3: negativos ANTES do piso ---
neg_antes = int((cobertura["DIAS_COBERTURA"] < 0).sum())
min_antes = cobertura["DIAS_COBERTURA"].replace(np.inf, np.nan).min()

# piso em 0 (Bug 3)
cobertura["DIAS_COBERTURA"] = np.where(cobertura["ESTOQUE_PROJ"] <= 0, 0, cobertura["DIAS_COBERTURA"])

neg_depois = int((cobertura["DIAS_COBERTURA"] < 0).sum())
min_depois = cobertura["DIAS_COBERTURA"].replace(np.inf, np.nan).min()
print(f"DIAS_COBERTURA negativos — ANTES do piso: {neg_antes:,} (mínimo {min_antes:.0f} dias)")
print(f"DIAS_COBERTURA negativos — DEPOIS do piso: {neg_depois}   (mínimo {min_depois:.0f} dias)")

DIAS_COBERTURA negativos — ANTES do piso: 18,201 (mínimo -720 dias)
DIAS_COBERTURA negativos — DEPOIS do piso: 0   (mínimo 0 dias)


**Por quê piso em 0?** A condição de ruptura já é capturada pelo `STATUS_ESTOQUE = "EM RUPTURA"`.
O número negativo só atrapalha ordenações e médias futuras. No **universo corrigido** (28.721
pares, já com o Bug 2) seriam **18.201** valores negativos sem o piso; no **output original com
bug** (25.330 pares) eram **14.939**. Em ambos os casos, o piso zera todos.

### Correção do Bug 1 — receita por par com todas as lojas
A versão com bug calculava a receita excluindo a loja 93. Como o skeleton inclui pares da loja
93, eles ficavam com `RECEITA_TOTAL` nula e **sumiam do ranking de reposição**. A célula compara
as duas formas de calcular.

In [7]:
# Receita por par — comparação BUGADA (sem loja 93) × CORRIGIDA (todas as lojas)
rec_bug = (v[v["COD_EMPRESA"] != LOJA_ATACADO]
           .groupby(["COD_EMPRESA","CODIGO"])["RECEITA"].sum()
           .reset_index().rename(columns={"RECEITA":"RECEITA_TOTAL"}))
rec_por_par = (v.groupby(["COD_EMPRESA","CODIGO"])["RECEITA"].sum()
               .reset_index().rename(columns={"RECEITA":"RECEITA_TOTAL"}))   # Bug 1 corrigido

# pares da loja 93 em ruptura
rupt93 = cobertura[(cobertura["COD_EMPRESA"]==LOJA_ATACADO) & (cobertura["ESTOQUE_PROJ"]<=0)][["COD_EMPRESA","CODIGO"]]
m_bug = rupt93.merge(rec_bug, on=["COD_EMPRESA","CODIGO"], how="left")
m_fix = rupt93.merge(rec_por_par, on=["COD_EMPRESA","CODIGO"], how="left")
print(f"Pares da loja 93 em ruptura: {len(rupt93):,}")
print(f"  com RECEITA preenchida — cálculo BUGADO (sem loja 93): {int((m_bug['RECEITA_TOTAL']>0).sum())}")
print(f"  com RECEITA preenchida — cálculo CORRIGIDO            : {int((m_fix['RECEITA_TOTAL']>0).sum())}")
print(f"  → {int((m_fix['RECEITA_TOTAL']>0).sum())} pares da loja 93 voltam ao ranking de prioridade")

Pares da loja 93 em ruptura: 2,232
  com RECEITA preenchida — cálculo BUGADO (sem loja 93): 0
  com RECEITA preenchida — cálculo CORRIGIDO            : 263
  → 263 pares da loja 93 voltam ao ranking de prioridade


**Por quê a receita deve incluir a loja 93?** A receita é um **fato do par** loja×produto.
A loja 93 vende (e muito): seus pares em ruptura — condicionadores, lavadoras, TVs — são os de
**maior receita histórica** da base. Zerá-los escondia exatamente os itens mais relevantes.
**263 pares** da loja 93 retornam ao ranking.

In [8]:
# Classificação de status + enriquecimento
def classifica_estoque(row):
    if row["ESTOQUE_PROJ"] <= 0:           return "EM RUPTURA"
    elif row["DIAS_COBERTURA"] <= 30:      return "CRÍTICO"
    elif row["DIAS_COBERTURA"] <= 90:      return "ATENÇÃO"
    elif row["DIAS_COBERTURA"] == np.inf:  return "SEM VENDA"
    else:                                  return "SAUDÁVEL"

cobertura["STATUS_ESTOQUE"] = cobertura.apply(classifica_estoque, axis=1)
cobertura = (cobertura
    .merge(rec_por_par, on=["COD_EMPRESA","CODIGO"], how="left")
    .merge(p[["CODIGO","DESCRICAO","NIVEL_1"]], on="CODIGO", how="left")
    .merge(l, on="COD_EMPRESA", how="left"))

print(cobertura["STATUS_ESTOQUE"].value_counts().to_string())

STATUS_ESTOQUE
EM RUPTURA    26140
SAUDÁVEL        955
SEM VENDA       662
CRÍTICO         573
ATENÇÃO         391


**Critérios de status:** *Em Ruptura* (estoque ≤ 0) → *Crítico* (≤ 30 dias) → *Atenção*
(31–90 dias) → *Saudável* (> 90 dias) → *Sem Venda* (sem demanda na rede física = capital imobilizado).

### Correção do Bug 4 — investigação dos outliers de preço
Produto **119959** (Adaptador Cx. Elét. 3/4) varia de R$3,68 a R$391,55 na mesma loja.
Hipótese: os lançamentos caros são vendas em **caixa** (`EMBALAGEM=1`). Se a conversão dessas
linhas fosse 1, seria bug (preço da caixa lançado como unitário).

In [9]:
# Diagnóstico do produto emblemático 119959
prod = v[v["CODIGO"]==119959]
diag119959 = prod.groupby("EMBALAGEM").agg(
    n=("PRECO_UNIT_MEDIO","count"),
    preco_min=("PRECO_UNIT_MEDIO","min"),
    preco_max=("PRECO_UNIT_MEDIO","max"),
    conversao=("CONVERSAO_VENDA_PARA_ARMAZENAGEM","max"),
)
print("Produto 119959 — preço × embalagem × conversão:")
print(diag119959.to_string())
print("\n→ EMBALAGEM=1 tem CONVERSAO=100 (caixa de 100). R$391,55 ÷ 100 ≈ R$3,92/unidade —")
print("  consistente com a venda unitária. A conversão ESTÁ aplicada: não é bug, é venda em caixa.")
diag119959

Produto 119959 — preço × embalagem × conversão:
              n  preco_min  preco_max  conversao
EMBALAGEM                                       
0          1424      3.045      4.410        1.0
1             5    362.145    391.545      100.0

→ EMBALAGEM=1 tem CONVERSAO=100 (caixa de 100). R$391,55 ÷ 100 ≈ R$3,92/unidade —
  consistente com a venda unitária. A conversão ESTÁ aplicada: não é bug, é venda em caixa.


,n,preco_min,preco_max,conversao
EMBALAGEM,,,,
0,1424,3.045,4.410,1.0
1,5,362.145,391.545,100.0


In [10]:
# 7 — Investigação estendida a TODOS os pares com CV de preço > 1
preco_stats = (v.groupby(["COD_EMPRESA","CODIGO"])["PRECO_UNIT_MEDIO"]
    .agg(PRECO_MIN="min", PRECO_MAX="max", PRECO_MEDIO="mean", PRECO_STD="std", N_LANC="count").reset_index())
preco_stats["CV"] = preco_stats["PRECO_STD"]/preco_stats["PRECO_MEDIO"]
outliers = preco_stats[preco_stats["CV"] > 1].copy()

emb1 = (v[v["EMBALAGEM"]==1].groupby(["COD_EMPRESA","CODIGO"])
        .agg(EMB1_N=("PRECO_UNIT_MEDIO","count"),
             EMB1_CONV_MAX=("CONVERSAO_VENDA_PARA_ARMAZENAGEM","max"),
             EMB1_PRECO_MAX=("PRECO_UNIT_MEDIO","max")).reset_index())
outliers = outliers.merge(emb1, on=["COD_EMPRESA","CODIGO"], how="left")
outliers["EMB1_N"] = outliers["EMB1_N"].fillna(0).astype(int)

def diagnostica_outlier(r):
    if r["EMB1_N"] > 0:
        conv = r["EMB1_CONV_MAX"]
        if pd.notna(conv) and conv > 1:
            unit = r["EMB1_PRECO_MAX"]/conv
            hip = (f"Preço alto = venda em embalagem/caixa (EMBALAGEM=1, "
                   f"CONVERSAO={conv:g}); preço por unidade de armazenagem ≈ "
                   f"R${unit:.2f}. Conversão presente (≠1) — não é o bug de "
                   f"conversão ausente.")
            return pd.Series({"HIPOTESE": hip, "EMBALAGEM_SUSPEITA":"não"})
        hip = ("EMBALAGEM=1 com CONVERSAO=1: preço de caixa lançado como "
               "unitário sem conversão — provável erro de cadastro (bug).")
        return pd.Series({"HIPOTESE": hip, "EMBALAGEM_SUSPEITA":"sim"})
    hip = ("Variação de preço sem mudança de embalagem (EMBALAGEM=0 em todos "
           "os lançamentos): repricing/desconto ao longo do período.")
    return pd.Series({"HIPOTESE": hip, "EMBALAGEM_SUSPEITA":"não"})

diag = outliers.apply(diagnostica_outlier, axis=1)
outliers = pd.concat([outliers, diag], axis=1).merge(p[["CODIGO","DESCRICAO"]], on="CODIGO", how="left")
outliers["PRECO_MIN"] = outliers["PRECO_MIN"].round(2)
outliers["PRECO_MAX"] = outliers["PRECO_MAX"].round(2)
outliers["CV"]        = outliers["CV"].round(3)
investigacao = (outliers.sort_values("CV", ascending=False)
    [["COD_EMPRESA","CODIGO","DESCRICAO","PRECO_MIN","PRECO_MAX","CV","HIPOTESE","EMBALAGEM_SUSPEITA"]]
    .reset_index(drop=True))

print(f"Pares com CV de preço > 1: {len(investigacao)}")
print("EMBALAGEM_SUSPEITA:", investigacao["EMBALAGEM_SUSPEITA"].value_counts().to_dict())
investigacao.head(8)

Pares com CV de preço > 1: 75
EMBALAGEM_SUSPEITA: {'não': 75}


,COD_EMPRESA,CODIGO,DESCRICAO,PRECO_MIN,PRECO_MAX,CV,HIPOTESE,EMBALAGEM_SUSPEITA
0,8,119959,ADAPTA.CX.ELET.AL 3/4 56251/052,3.68,362.14,4.628,Preço alto = venda em embalagem/caixa (EMBALAG...,não
1,4,119959,ADAPTA.CX.ELET.AL 3/4 56251/052,3.68,391.54,4.573,Preço alto = venda em embalagem/caixa (EMBALAG...,não
2,2,119959,ADAPTA.CX.ELET.AL 3/4 56251/052,3.04,391.54,4.118,Preço alto = venda em embalagem/caixa (EMBALAG...,não
3,3,369633,CANALETA S/DIV. ADESIV. 10X10 2MT BR,7.25,745.40,3.889,Preço alto = venda em embalagem/caixa (EMBALAG...,não
4,3,270605,CX.ELET.PLS 4X4 AM OCT0G. 57500/004,6.51,371.60,3.660,Preço alto = venda em embalagem/caixa (EMBALAG...,não
5,5,270605,CX.ELET.PLS 4X4 AM OCT0G. 57500/004,7.25,371.60,2.933,Preço alto = venda em embalagem/caixa (EMBALAG...,não
6,4,421744,PRATO SMES.CER.19CM SICILIANO 92956,10.49,299.14,2.380,Preço alto = venda em embalagem/caixa (EMBALAG...,não
7,4,326263,SUPORTE 4X2 3MOD. BR 850200DUALE UP,1.58,50.30,2.014,Preço alto = venda em embalagem/caixa (EMBALAG...,não


**Achado (Bug 4):** dos **75** pares com CV > 1, **nenhum** é o bug de conversão ausente.
Em todos, os preços altos são vendas em **embalagem/caixa** (`EMBALAGEM=1`) com a
`CONVERSAO_VENDA_PARA_ARMAZENAGEM` corretamente preenchida (12, 20, 25, 60, 100…). O preço por
unidade de armazenagem fica coerente com a venda unitária, e `QTD_ARMAZENAGEM`/`RECEITA` já saem
corretas. **Conclusão: não é bug** — é venda em caixa legítima. Documentado em
`investigacao_outliers_preco.csv`.

In [11]:
# 8 — Top 15 pares críticos por receita (pós-correção): a loja 93 reaparece
top = (cobertura[cobertura["STATUS_ESTOQUE"].isin(["EM RUPTURA","CRÍTICO"])]
    .sort_values("RECEITA_TOTAL", ascending=False)
    [["COD_EMPRESA","CD_CIDADE","CODIGO","DESCRICAO","NIVEL_1","ESTOQUE_PROJ","DIAS_COBERTURA","RECEITA_TOTAL"]]
    .head(15).reset_index(drop=True))
top

,COD_EMPRESA,CD_CIDADE,CODIGO,DESCRICAO,NIVEL_1,ESTOQUE_PROJ,DIAS_COBERTURA,RECEITA_TOTAL
0,93,ALHANDRA,467774,COND.SPLIT 9000 COND.S3UQ09 INV. 143,D - ELETROS,-5433.0,0.0,1.090942e+07
1,93,ALHANDRA,467770,COND.SPLIT 9000 COND.S3UQ09 INV. 141,D - ELETROS,-4611.0,0.0,8.555417e+06
2,93,ALHANDRA,470176,LAV.ROUPA 13.0KG BWK13AB BR M220,D - ELETROS,-2474.0,0.0,7.136044e+06
3,93,ALHANDRA,454103,REFRIG. 2P 386L FF CRM44ABB BR M220,D - ELETROS,-1916.0,0.0,5.967726e+06
4,93,ALHANDRA,467773,COND.SPLIT 9000 EVAP.S3NQ09 INV. 143,D - ELETROS,-5433.0,0.0,5.375682e+06
5,93,ALHANDRA,467769,COND.SPLIT 9000 EVAP.S3NQ09 INV. 141,D - ELETROS,-4611.0,0.0,4.533555e+06
6,93,ALHANDRA,467736,COND.SPLIT 12000 COND+EVAP. ACST12F,D - ELETROS,-644.0,0.0,3.901232e+06
7,93,ALHANDRA,481978,COND.SPLIT COND 12000 HIQE12 INV.202,D - ELETROS,-2544.0,0.0,3.839458e+06
8,93,ALHANDRA,466938,TV LED 50 UHD SMT 4K 50UR8750PSA,R - ELETRONICOS,-1424.0,0.0,3.611342e+06
9,3,SALVADOR,432048,MASSA CORRIDA PVA CORAL PLS 25KG,S - TINTAS E QUIMICOS,-35862.0,0.0,3.047831e+06


**Leitura do ranking:** o topo agora é dominado por pares da **loja 93** (condicionadores,
lavadoras, TVs) — exatamente os que o Bug 1 escondia. Sem a correção, o plano de reposição
ignoraria os itens de maior receita da base.

In [12]:
# 9 — Salvar saídas
df.to_parquet(PROCESSED / "estoque_projetado.parquet", index=False)
cobertura.to_parquet(PROCESSED / "cobertura_estoque.parquet", index=False)
cobertura.to_csv(OUT / "cobertura_estoque.csv", index=False, encoding="utf-8-sig")
investigacao.to_csv(OUT / "investigacao_outliers_preco.csv", index=False, encoding="utf-8-sig")
print("Salvos: estoque_projetado.parquet, cobertura_estoque.parquet,")
print("        outputs/etapa2/cobertura_estoque.csv, outputs/etapa2/investigacao_outliers_preco.csv")

Salvos: estoque_projetado.parquet, cobertura_estoque.parquet,
        outputs/etapa2/cobertura_estoque.csv, outputs/etapa2/investigacao_outliers_preco.csv


## Resumo dos KPIs validados — Etapa 2 (pós-correção)

| Indicador | Antes | Depois |
|---|---|---|
| Pares loja×produto na cobertura | 25.330 | **28.721** (+3.391) |
| Receita rastreada na cobertura | — | **+R$ 87,5M** (pares Bug 2) |
| Loja 93 em ruptura com receita no ranking | 0 | **263** |
| `DIAS_COBERTURA` negativos | 14.939 | **0** |
| Pares com CV de preço > 1 investigados | — | **75** (0 bugs, 75 explicados) |

| Status (dez/2025) | Pares | % |
|---|---|---|
| Em Ruptura | 26.140 | 91,0% |
| Crítico | 573 | 2,0% |
| Atenção | 391 | 1,4% |
| Saudável | 955 | 3,3% |
| Sem Venda | 662 | 2,3% |

As 4 correções estão detalhadas no dashboard executivo
`outputs/relatorio_qualidade_dados.html`.

## Premissas, limitações conhecidas e próximos passos

Pontos que eu sinalizaria a um revisor antes de tratar os números como definitivos:

- **`VENDA_MEDIA_MES` é a média dos meses *com* venda, não dos 24 meses.** Para itens de venda
  intermitente isso **superestima a velocidade** e **subestima os dias de cobertura** — empurrando
  itens para "Crítico". Escolha herdada da Etapa 2 original, mantida por consistência; comparar com
  a variante "volume total ÷ 24 meses" na próxima iteração.
- **A ruptura de ~91% não é ruptura física real de 91% da rede.** Reflete a *ausência de registros
  de reposição* (~88% dos SKUs vendem sem compra registrada — provável transferência entre lojas
  não capturada) + estoque inicial nulo tratado como 0. Serve para **priorização relativa**, não
  como medida absoluta de disponibilidade.
- **Loja 93 (atacado)** entra nos cálculos de receita e estoque (são fatos do par), mas é excluída
  da *referência de consumo da rede física* na cobertura em dias — pode gerar pares da loja 93 com
  receita alta e status "Sem Venda" (esperado, não erro).

**Próximos passos:** (1) cobertura com média sobre 24 meses + sensibilidade do status;
(2) incorporar transferências entre lojas para fechar o gap dos SKUs sem compra;
(3) decompor a queda de receita de 2025 por categoria e canal (atacado × rede).